In [ ]:
import requests
import json
from tqdm import tqdm

def get_metadata_current(articles, output_file="metadata.ndjson"):
  # retrieve metadata from xtools.wmcloud.org
  with open(output_file, 'w') as f:
    for a in tqdm(articles, desc="Fetching metadata"):            # any large number ─ loop can be slow
      url_prose = 'https://xtools.wmcloud.org/api/page/prose/en.wikipedia.org/' + a
      response_prose = requests.get(url_prose)
      if response_prose.status_code == 200:
        response_prose_json = response_prose.json()
        url_pageinfo = 'https://xtools.wmcloud.org/api/page/pageinfo/en.wikipedia.org/' + a
        response_pageinfo = requests.get(url_pageinfo)
        response_pageinfo_json = response_pageinfo.json()
        url_links = 'https://xtools.wmcloud.org/api/page/links/en.wikipedia.org/' + a
        response_links = requests.get(url_links)
        response_links_json = response_links.json()
        if 'characters' in response_prose_json:
          doc_metadata = {
              "article": response_prose_json["page"],
              "characters": response_prose_json["characters"],
              "words": response_prose_json["words"],
              "sections": response_prose_json["sections"],
              "unique_references": response_prose_json["unique_references"],
              "watchers": response_pageinfo_json["watchers"],
              "revisions": response_pageinfo_json["revisions"],
              "editors": response_pageinfo_json["editors"],
              "created_at": response_pageinfo_json["created_at"],
              "modified_at": response_pageinfo_json["modified_at"],
              "links_ext_count": response_links_json["links_ext_count"],
              "links_out_count": response_links_json["links_out_count"],
              "links_in_count": response_links_json["links_in_count"],
              "redirects_count": response_links_json["redirects_count"]
            }
          # Write each doc_metadata as a JSON line and flush immediately
          f.write(json.dumps(doc_metadata) + '\n')
          f.flush()

In [ ]:
articles_file = "articles.json"
articles_dict = json.load(open(articles_file))
article_titles = list(articles_dict.keys())
# print(len(article_titles))
metadata = get_metadata_current(article_titles)
